# Lesson15: LangChainの主要モジュール6【Callbacks】

## 事前準備
シークレット機能でOpenAI APIのAPIキーを設定し、以下2つのコードを実行しましょう。

In [ ]:
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

In [ ]:
!pip install langchain==0.3.26 openai==1.91.0 langchain-community==0.3.26 httpx==0.28.1

## 記述用

In [ ]:
from langchain.callbacks.base import BaseCallbackHandler
from typing import Dict, Any, List, Union, Sequence
from langchain.schema import BaseMessage, LLMResult, AgentAction, AgentFinish, Document

class CustomCallbackHandler(BaseCallbackHandler):

    def on_chat_model_start(self, serialized: Dict[str, Any], messages: List[List[BaseMessage]], **kwargs: Any):
        """チャットモデルの処理開始時"""

    def on_llm_new_token(self, token: str, **kwargs: Any):
        """新規トークン生成時"""

    def on_llm_error(self, error: Union[Exception, KeyboardInterrupt], **kwargs: Any):
        """LLMのエラー発生時"""

    def on_llm_end(self, response: LLMResult, **kwargs: Any):
        """LLMの処理終了時"""

    def on_chain_start(self, serialized: Dict[str, Any], inputs: Dict[str, Any], **kwargs: Any):
        """Chainsの処理開始時"""

    def on_chain_error(self, error: Union[Exception, KeyboardInterrupt], **kwargs: Any):
        """Chainsのエラー発生時"""

    def on_chain_end(self, outputs: Dict[str, Any], **kwargs: Any):
        """Chainsの処理終了時"""

    def on_retriever_start(self, serialized: Dict[str, Any], query: str, **kwargs: Any):
        """Retrievalの処理開始時"""

    def on_retriever_error(error: BaseException, **kwargs: Any):
        """Retrievalのエラー発生時"""

    def on_retriever_end(documents: Sequence[Document], **kwargs: Any):
        """Retrievalの処理終了時"""

    def on_tool_start(self, serialized: Dict[str, Any], input_str: str, **kwargs: Any):
        """エージェントに渡したToolの動作開始時"""

    def on_tool_error(self, error: Union[Exception, KeyboardInterrupt], **kwargs: Any):
        """エージェントに渡したTool動作中のエラー発生時"""

    def on_tool_end(self, output: str, **kwargs: Any):
        """エージェントに渡したToolの動作終了時"""

    def on_agent_action(self, action: AgentAction, **kwargs: Any):
        """エージェントの特定アクション開始時"""

    def on_agent_finish(self, finish: AgentFinish, **kwargs: Any):
        """エージェントの特定アクション終了時"""




In [ ]:
class CustomCallbackHandler(BaseCallbackHandler):

    def on_chat_model_start(self, serialized: Dict[str, Any], messages: List[List[BaseMessage]], **kwargs: Any):
        """チャットモデルの処理開始時"""
        print("処理開始")
        print("======================")

    def on_llm_end(self, response: LLMResult, **kwargs: Any):
        """LLMの処理終了時"""
        print("\n======================")
        print("処理終了")

In [ ]:
from langchain.chat_models import ChatOpenAI
from langchain.schema import SystemMessage, HumanMessage
from langchain.callbacks.streaming_stdout import StreamingStdOutCallbackHandler

llm = ChatOpenAI(
    model_name="gpt-4o-mini",
    temperature=0.5,
    streaming=True,
    callbacks=[CustomCallbackHandler(), StreamingStdOutCallbackHandler()]
)

messages = [
    SystemMessage(content="You are a helpful assistant."),
    HumanMessage(content="なぜ東京スカイツリーの高さは634mなんですか？"),
]

result = llm(messages)

In [ ]:
!pip install loguru==0.7.3

In [ ]:
from loguru import logger

from langchain.callbacks import FileCallbackHandler
from langchain.chat_models import ChatOpenAI
from langchain.prompts import PromptTemplate
from langchain import LLMChain

logfile = "output.log"

logger.add(logfile)
handler = FileCallbackHandler(logfile)

In [ ]:
template = """
以下の地名の、おすすめの観光スポットを3つ教えてください。

地名：{place}
"""

prompt = PromptTemplate(
    input_variables=["place"],
    template=template,
)

llm = ChatOpenAI(model_name="gpt-4o-mini", temperature=0.5)

chain = LLMChain(prompt=prompt, llm=llm, callbacks=[handler], verbose=True)
result = chain.run(place="東京都")

logger.info(result)
print(result)